# Llama Retroactive Interference Experiments

**Purpose**: Test Llama models with Level 200+ interference (requires full 128K context)

**Setup**:
1. Runtime → Change runtime type → T4 GPU
2. Upload dataset file: `interleaved_dataset.json`
3. Upload response parser: `category_response_parser.py`
4. Run all cells

**What this notebook does**:
- Loads Llama with 4-bit quantization (fits in free T4 GPU)
- Tests Levels 200, 300, 400 (requires 128K context)
- Parses responses properly
- Saves results as JSON

## Cell 1: Install Dependencies

In [ ]:
!pip install -U transformers bitsandbytes accelerate -q
print("✅ Dependencies installed!")

## Cell 2: Login to Hugging Face

In [ ]:
from huggingface_hub import login
import os

# Option 1: Interactive login
login(new_session=False)

# Option 2: Set token directly (uncomment and replace)
# os.environ['HF_TOKEN'] = 'hf_your_token_here'

# Set memory allocation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## Cell 3: Upload Required Files

Upload these files from your local machine:
1. `interleaved_dataset.json` (from `synthetic_dataset_category/` folder)
2. `category_response_parser.py` (from `datasets/` folder)

In [ ]:
from google.colab import files
import os

print("Please upload:")
print("1. interleaved_dataset.json")
print("2. category_response_parser.py")
print("\nClick 'Choose Files' below...")

uploaded = files.upload()

# Verify files
if 'interleaved_dataset.json' in uploaded:
    print("✅ Dataset uploaded")
else:
    print("❌ Missing: interleaved_dataset.json")

if 'category_response_parser.py' in uploaded:
    print("✅ Parser uploaded")
else:
    print("❌ Missing: category_response_parser.py")

## Cell 4: Load Response Parser

This is the same parser used in your main experiments

In [ ]:
import sys
import importlib.util

# Load the parser module
spec = importlib.util.spec_from_file_location("category_response_parser", "category_response_parser.py")
parser_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parser_module)

CategoryResponseParser = parser_module.CategoryResponseParser

print("✅ Response parser loaded!")

## Cell 5: Load Dataset

Load the interleaved dataset and define helper functions

In [ ]:
import json
import random

# Load dataset
with open('interleaved_dataset.json', 'r') as f:
    dataset = json.load(f)

print(f"Dataset loaded:")
print(f"  Categories: {len(dataset['baselines'])}")
print(f"  Levels: {dataset['levels']}")

def create_interleaved_prompt(dataset, categories, interference_level):
    """
    Create an interleaved prompt with TRUE random interleaving.
    This matches the implementation in your main experiment code.
    """
    baselines = dataset['baselines']
    interference_data = dataset['interference']

    # Get baseline values
    baseline_pairs = {}
    for category in categories:
        baseline_list = baselines[category]
        baseline_idx = random.randint(0, len(baseline_list) - 1)
        baseline_pairs[category] = baseline_list[baseline_idx]

    # Create sequence with TRUE random interleaving
    sequence = []
    baseline_positions = {}

    for category in categories:
        # Add baseline (first occurrence)
        sequence.append((category, baseline_pairs[category]))

        # Add interference items
        interference_items = interference_data[str(interference_level)][category]
        for item in interference_items:
            sequence.append((category, item))

    # Shuffle to create TRUE random interleaving
    random.shuffle(sequence)

    # Record baseline positions AFTER shuffling
    for i, (cat, val) in enumerate(sequence):
        if cat not in baseline_positions and val == baseline_pairs[cat]:
            baseline_positions[cat] = i

    # Build prompt
    prompt_parts = []
    prompt_parts.append("Below is a stream of category-value pairs in the format <category>: <value>.")
    prompt_parts.append("Your task is to identify the INITIAL value (the first occurrence) for each category.\n")

    for cat, val in sequence:
        prompt_parts.append(f"{cat}: {val}")

    prompt_parts.append("\nFor each of the following categories, what was the INITIAL value (i.e., the value from the FIRST occurrence of that category)?\n")
    for cat in categories:
        prompt_parts.append(f"- {cat}")

    prompt_parts.append("\nProvide your answer in this format:")
    prompt_parts.append("The initial value of <category> is <value>.")

    prompt = "\n".join(prompt_parts)

    return {
        'prompt': prompt,
        'expected_answers': baseline_pairs,
        'baseline_positions': baseline_positions,
        'sequence_length': len(sequence),
        'categories': categories
    }

# Test with small example
test_categories = list(dataset['baselines'].keys())[:4]
test_prompt_data = create_interleaved_prompt(dataset, test_categories, 3)

print(f"\n✅ Prompt creation working!")
print(f"  Test categories: {len(test_categories)}")
print(f"  Test sequence length: {test_prompt_data['sequence_length']}")

## Cell 6: Load Llama Model with 4-bit Quantization

**This will take 2-3 minutes** to download and load the model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Model selection (change this to test different models)
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"  # 128K context
# MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"  # 128K context, smaller

print(f"Loading model: {MODEL_NAME}")
print("This will take 2-3 minutes...\n")

# Check GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n")

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load model with 4-bit quantization
print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"\n✅ Model loaded successfully!")
print(f"  Device: {model.device}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"  Context length: 128K tokens")

## Cell 7: Define Generation Function

In [ ]:
def generate_response(prompt, max_tokens=8000, temperature=0.0):
    """
    Generate response from Llama model.

    Args:
        prompt: Input prompt string
        max_tokens: Maximum tokens to generate
        temperature: Sampling temperature (0.0 = deterministic)

    Returns:
        Generated response string
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs.input_ids.shape[1]

    print(f"  Input tokens: {input_length}")
    print(f"  Max output tokens: {max_tokens}")
    print(f"  Generating...", end='', flush=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature if temperature > 0 else None,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the generated tokens (not the input)
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    print(" Done!")
    print(f"  Output tokens: {len(outputs[0]) - input_length}")
    print(f"  Response length: {len(response)} chars")

    return response

# Test with simple prompt
print("Testing generation with simple prompt:")
test_response = generate_response("What is 2+2? Answer with just the number.", max_tokens=10)
print(f"\nResponse: {test_response}")
print("\n✅ Generation function working!")

## Cell 8: Run Single Experiment

Test a single level first to make sure everything works

In [ ]:
def run_experiment(dataset, model_name, interference_level, sample_size=46, max_tokens=8000):
    """
    Run a single interference experiment.

    Args:
        dataset: Loaded dataset dict
        model_name: Model name for logging
        interference_level: Level of interference (3, 10, 50, 100, 200, etc.)
        sample_size: Number of categories to test
        max_tokens: Maximum tokens for generation

    Returns:
        Results dictionary
    """
    print("="*80)
    print(f"EXPERIMENT: {model_name}, Level {interference_level}")
    print("="*80)

    # Get categories
    all_categories = list(dataset['baselines'].keys())
    categories = all_categories[:sample_size]

    print(f"Testing {len(categories)} categories with {interference_level} updates each")

    # Create prompt
    prompt_data = create_interleaved_prompt(dataset, categories, interference_level)
    prompt = prompt_data['prompt']
    expected_answers = prompt_data['expected_answers']
    baseline_positions = prompt_data['baseline_positions']

    print(f"Sequence length: {prompt_data['sequence_length']}")

    # Generate response
    print(f"\nCalling LLM...")
    response = generate_response(prompt, max_tokens=max_tokens)

    # Parse response
    print(f"\nParsing response...")
    extracted_values = CategoryResponseParser.parse_batch_response(response, categories)

    # Evaluate results
    results = []
    correct_count = 0
    missing_count = 0

    for category in categories:
        extracted = extracted_values.get(category)
        expected = expected_answers[category]
        baseline_pos = baseline_positions[category]

        is_correct = False
        is_missing = False

        if extracted is None:
            is_missing = True
            missing_count += 1
        elif extracted.lower() == expected.lower():
            is_correct = True
            correct_count += 1

        results.append({
            "category": category,
            "expected": expected,
            "extracted": extracted,
            "correct": is_correct,
            "missing": is_missing,
            "baseline_position": baseline_pos
        })

    # Calculate metrics
    accuracy = (correct_count / len(categories)) * 100

    print(f"\nResults:")
    print(f"  Correct: {correct_count}/{len(categories)} ({accuracy:.2f}%)")
    print(f"  Missing: {missing_count}/{len(categories)} ({missing_count/len(categories)*100:.2f}%)")

    # Show errors
    errors = [r for r in results if not r['correct']]
    if errors:
        print(f"\nExample errors (first 5):")
        for error in errors[:5]:
            print(f"  {error['category']}:")
            print(f"    Expected: {error['expected']}")
            print(f"    Extracted: {error['extracted']}")

    return {
        "model": model_name,
        "interference_level": interference_level,
        "sample_size": len(categories),
        "accuracy": accuracy,
        "correct_count": correct_count,
        "total_count": len(categories),
        "missing_count": missing_count,
        "sequence_length": prompt_data['sequence_length'],
        "results": results,
        "full_response": response  # Keep full response for debugging
    }

# Test with Level 100 first (smaller, faster)
print("Testing with Level 100 (should work with HF Router too)...\n")
test_result = run_experiment(
    dataset=dataset,
    model_name=MODEL_NAME.split('/')[-1],
    interference_level=100,
    sample_size=46,
    max_tokens=8000
)

print("\n✅ Experiment complete!")

## Cell 9: Run Level 200+ Experiments

**This is the main goal** - test levels that exceed HF Router limits

In [ ]:
import datetime

# Configuration
LEVELS_TO_TEST = [200, 300, 400]  # These require full 128K context
SAMPLE_SIZE = 46  # All categories
MAX_TOKENS = 8000  # Enough for 46 categories

# Run experiments
all_results = []

for level in LEVELS_TO_TEST:
    print(f"\n\n{'='*80}")
    print(f"STARTING LEVEL {level}")
    print(f"{'='*80}\n")

    try:
        result = run_experiment(
            dataset=dataset,
            model_name=MODEL_NAME.split('/')[-1],
            interference_level=level,
            sample_size=SAMPLE_SIZE,
            max_tokens=MAX_TOKENS
        )
        all_results.append(result)

        print(f"\n✅ Level {level} complete!")

    except Exception as e:
        print(f"\n❌ ERROR at Level {level}: {e}")
        all_results.append({
            "model": MODEL_NAME.split('/')[-1],
            "interference_level": level,
            "error": str(e)
        })

# Summary
print("\n\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\nModel: {MODEL_NAME}")
print(f"Levels tested: {LEVELS_TO_TEST}")
print(f"\nAccuracy by level:")
for result in all_results:
    if 'accuracy' in result:
        level = result['interference_level']
        accuracy = result['accuracy']
        correct = result['correct_count']
        total = result['total_count']
        missing = result['missing_count']
        print(f"  Level {level:3d}: {accuracy:5.2f}% ({correct}/{total}) [Missing: {missing}]")
    else:
        print(f"  Level {result['interference_level']:3d}: ERROR - {result.get('error', 'Unknown')}")

print("\n✅ All experiments complete!")

## Cell 10: Save Results

In [ ]:
import json
from datetime import datetime

# Prepare results for saving
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_short = MODEL_NAME.split('/')[-1].lower().replace('-', '_')
levels_str = '_'.join(map(str, LEVELS_TO_TEST))
filename = f"{model_short}_levels_{levels_str}_colab_{timestamp}.json"

# Remove full_response to save space (optional)
results_for_saving = []
for result in all_results:
    result_copy = result.copy()
    if 'full_response' in result_copy:
        # Keep only first 1000 chars of response
        result_copy['batch_response'] = result_copy['full_response'][:1000]
        del result_copy['full_response']
    results_for_saving.append(result_copy)

output_data = {
    "experiment": "interleaved_retroactive_interference",
    "model": MODEL_NAME,
    "levels": LEVELS_TO_TEST,
    "sample_size": SAMPLE_SIZE,
    "timestamp": timestamp,
    "environment": "google_colab_gpu",
    "quantization": "4bit_nf4",
    "results": results_for_saving
}

# Save to file
with open(filename, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"✅ Results saved to: {filename}")

# Download the file
files.download(filename)

print(f"\n📥 File downloaded! Upload this to your results_interleaved/ folder.")

## Cell 11: Visualize Results (Optional)

In [ ]:
import matplotlib.pyplot as plt

# Extract data
levels = []
accuracies = []
missing_rates = []

for result in all_results:
    if 'accuracy' in result:
        levels.append(result['interference_level'])
        accuracies.append(result['accuracy'])
        missing_rate = (result['missing_count'] / result['total_count']) * 100
        missing_rates.append(missing_rate)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(levels, accuracies, marker='o', linewidth=2, markersize=8)
ax1.set_xlabel('Interference Level', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title(f'Accuracy vs Interference Level\n{MODEL_NAME.split("/")[-1]}', fontsize=14)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 105)

# Missing rate plot
ax2.plot(levels, missing_rates, marker='s', linewidth=2, markersize=8, color='orange')
ax2.set_xlabel('Interference Level', fontsize=12)
ax2.set_ylabel('Missing Rate (%)', fontsize=12)
ax2.set_title(f'Missing Responses vs Interference Level\n{MODEL_NAME.split("/")[-1]}', fontsize=14)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(f'{model_short}_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved!")
files.download(f'{model_short}_results.png')

## Summary

**What you tested**:
- Levels 200, 300, 400 with full 128K context
- All 46 categories
- Same prompt format and parsing as main experiments

**Expected results**:
- Level 200: Should work (requires ~60K tokens)
- Level 300: Should work (requires ~85K tokens)
- Level 400: Should work (requires ~110K tokens)

**Files downloaded**:
1. `{model}_levels_{levels}_colab_{timestamp}.json` - Results
2. `{model}_results.png` - Visualization

**Next steps**:
1. Upload JSON results to your `results_interleaved/` folder
2. Compare with HF Router results (Levels 3-100)
3. Analyze the full context vs limited context performance